# Black Cat Syndrome - Does the penalty hold?

Confounder analysis testing whether black cats really wait longer to be adopted, or whether
the naive gap is explained by composition (age, breed, location, etc.).

**Dataset:** `../data/adopted_and_adoptable_cats.csv` (~1.4M rows; 841k adopted cats with valid `days_to_adopt`, 2020–2025).

Checks
1. Is the `days_to_adopt == 0` spike an artifact that manufactures/masks the effect?
2. Does **age composition** explain the gap? (the prime confounder)
3. Are black cats overrepresented in the still-waiting pool? (model-independent)
4. Regression: does the black effect survive **all** controls jointly?


In [22]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
pd.set_option('display.width', 140)

csv = '../data/adopted_and_adoptable_cats.csv'
cols = ['status','days_to_adopt','days_in_shelter','color_category','color','age','size','coat',
        'breeds_mixed','spayed_neutered','photo_count','stateQ_grouped','published_year',
        'status_changed_at','published_at']
df = pd.read_csv(csv, usecols=cols, low_memory=False)

In [23]:
adopted = df[df['status']=='adopted'].copy()
adopted['days_to_adopt'] = pd.to_numeric(adopted['days_to_adopt'], errors='coerce')
adopted = adopted[adopted['days_to_adopt'].notna()]

# cohort: solid Black vs Non-Black
def cohort(s):
    if s == 'Black': return 'Black'
    if s == 'Unknown' or pd.isna(s): return 'Unknown'
    return 'Non-Black'
adopted['cohort'] = adopted['color_category'].map(cohort)
print(f'adopted cats with valid days_to_adopt: {len(adopted):,}')
adopted['days_to_adopt'].describe(percentiles=[.1,.25,.5,.75,.9]).round(1)

adopted cats with valid days_to_adopt: 841,776


count    841776.0
mean         45.0
std          96.0
min           0.0
10%           0.0
25%           4.0
50%          16.0
75%          45.0
90%         108.0
max        1991.0
Name: days_to_adopt, dtype: float64

## Check 1 - the `days_to_adopt == 0` spike

>10% of adopted cats sit at exactly 0 days. If that mass differs by color it can fabricate or
hide the effect. Here we show it is **concentrated in `Unknown`-color records** and is equal
(~5%) for Black vs Non-Black, so it does not bias the comparison. All later steps use `days >= 1`.

In [24]:
z = adopted['days_to_adopt'] == 0
print(f"share at exactly 0 days: {z.mean():.1%}  (n={z.sum():,})")
same = adopted['status_changed_at'].astype(str) == adopted['published_at'].astype(str)
print(f"of 0-day rows, status_changed_at == published_at: {same[z].mean():.1%}")
print(f"of 0-day rows, status_changed_at is null: {adopted.loc[z,'status_changed_at'].isna().mean():.1%}")

print("\n0-day rate by cohort:")
print(adopted.groupby('cohort')['days_to_adopt'].apply(lambda s: (s==0).mean()).round(3))

share at exactly 0 days: 13.5%  (n=113,565)
of 0-day rows, status_changed_at == published_at: 1.5%
of 0-day rows, status_changed_at is null: 0.0%

0-day rate by cohort:
cohort
Black        0.049
Non-Black    0.048
Unknown      0.434
Name: days_to_adopt, dtype: float64


## Check 2 - age composition (the prime confounder)

If black cats skew older, the naive gap is age, not color.

Result: black cats are **not** older (if anything slightly younger), and the penalty **persists
within every age band** - so age does not explain it.

In [28]:
print("age mix WITHIN each cohort (row %):")
print(pd.crosstab(adopted['cohort'], adopted['age'], normalize='index').round(3))
print("\ncohort share WITHIN each age band:")
print(pd.crosstab(adopted['age'], adopted['cohort'], normalize='index').round(3))

work = adopted[(adopted['days_to_adopt']>=1) & (adopted['cohort'].isin(['Black','Non-Black']))].copy()
print(f"\n(days>=1, Unknown excluded, n={len(work):,})")
print("\nNAIVE gap (days_to_adopt):")
print(work.groupby('cohort')['days_to_adopt'].agg(['count','median','mean']).round(1))

print("\nSTRATIFIED within age band (median days):")
strat = work.groupby(['age','cohort'])['days_to_adopt'].median().unstack()
strat['gap (Black - NonBlack)'] = strat['Black'] - strat['Non-Black']
print(strat.round(1))

age mix WITHIN each cohort (row %):
age        Adult   Baby  Senior  Young
cohort                                
Black      0.234  0.519   0.026  0.221
Non-Black  0.257  0.497   0.029  0.217
Unknown    0.428  0.263   0.103  0.206

cohort share WITHIN each age band:
cohort  Black  Non-Black  Unknown
age                              
Adult   0.174      0.496    0.331
Baby    0.249      0.619    0.132
Senior  0.123      0.361    0.516
Young   0.221      0.565    0.215

(days>=1, Unknown excluded, n=621,218)

NAIVE gap (days_to_adopt):
            count  median  mean
cohort                         
Black      172447    23.0  55.9
Non-Black  448771    20.0  50.0

STRATIFIED within age band (median days):
cohort  Black  Non-Black  gap (Black - NonBlack)
age                                             
Adult    31.0       27.0                     4.0
Baby     19.0       16.0                     3.0
Senior   35.0       31.0                     4.0
Young    28.0       23.0                     

## Check 3 - black share of still-waiting vs adopted pool

Model-independent evidence. Black cats overrepresented among still-waiting (`adoptable`) relative
to their adopted flow => direct penalty signal, and a hint the adopted-only estimate *understates* it.

In [29]:
adoptable = df[df['status']=='adoptable'].copy()
adoptable['cohort'] = adoptable['color_category'].map(cohort)
for name, frame in [('ADOPTED', adopted), ('ADOPTABLE (still waiting)', adoptable)]:
    known = frame[frame['cohort'].isin(['Black','Non-Black'])]
    print(f"{name:28s} black share of known-color: {(known['cohort']=='Black').mean():.1%}  (n={len(known):,})")

ADOPTED                      black share of known-color: 27.8%  (n=652,758)
ADOPTABLE (still waiting)    black share of known-color: 33.1%  (n=52,560)


## Check 4 - regression: does the effect survive all controls?

Outcome = `log1p(days_to_adopt)`; adopted cats, `days >= 1`, `Unknown` excluded.
Black effect does **not** shrink with controls — it grows slightly (opposite of black-*dog* syndrome).

Note: `photo_count` is endogenous (shelters add photos to hard-to-place cats), included only as a
sensitivity check. `adopted_year` is all-null in the data, so `published_year` is the time control.

Do black cats have to wait longer at the shelter just because they're black?
Control test answers this question: "Between two cats who are twins in every way except colour, does the black one still wait longer?"

In [ ]:
reg = adopted[adopted['days_to_adopt']>=1].copy()
reg = reg[reg['cohort'].isin(['Black','Non-Black'])]
reg['black']  = (reg['cohort']=='Black').astype(int)
reg['ldays']  = np.log1p(reg['days_to_adopt'])
reg['photo_count'] = pd.to_numeric(reg['photo_count'], errors='coerce').fillna(0)
for c in ['age','size','coat','breeds_mixed','spayed_neutered','stateQ_grouped','published_year']:
    reg[c] = reg[c].astype(str)

# run OLS regression
def run(formula, label):
    """
    Run OLS regression to fit a best-fit line predicting wait time from colour plus all other control variables
    to tell how much the black flag adds waiting time on its own.
    """
    m = smf.ols(formula, data=reg).fit()
    b, se, pv = m.params['black'], m.bse['black'], m.pvalues['black']
    print(f"{label:52s} coef={b:+.4f} SE={se:.4f} p={pv:.1e}  =>  {(np.exp(b)-1)*100:+.1f}% longer  (R2={m.rsquared:.3f})")
    return m

print(f"baseline medians -> Black {reg[reg.black==1].days_to_adopt.median():.0f}d  "
      f"Non-Black {reg[reg.black==0].days_to_adopt.median():.0f}d   n={len(reg):,}\n")
CTRL = "C(age) + C(size) + C(coat) + C(breeds_mixed) + C(spayed_neutered) + C(stateQ_grouped) + C(published_year)"
run("ldays ~ black", "M0: raw (no controls)")
run("ldays ~ black + C(age)", "M1: + age")
run(f"ldays ~ black + {CTRL}", "M2: + all controls (no photo)")
m3 = run(f"ldays ~ black + {CTRL} + photo_count", "M3: M2 + photo_count (endogenous)")

print(f"\nmean photo_count: Black {reg[reg.black==1].photo_count.mean():.2f}  "
      f"Non-Black {reg[reg.black==0].photo_count.mean():.2f}")

baseline medians -> Black 23d  Non-Black 20d   n=621,218

M0: raw (no controls)                                coef=+0.1269 SE=0.0036 p=4.9e-274  =>  +13.5% longer  (R2=0.002)
M1: + age                                            coef=+0.1408 SE=0.0035 p=0.0e+00  =>  +15.1% longer  (R2=0.038)
M2: + all controls (no photo)                        coef=+0.1524 SE=0.0035 p=0.0e+00  =>  +16.5% longer  (R2=0.083)
M3: M2 + photo_count (endogenous)                    coef=+0.1574 SE=0.0034 p=0.0e+00  =>  +17.0% longer  (R2=0.105)

mean photo_count: Black 2.99  Non-Black 3.06


This concludes that the gap didnt shrink as we controlled for all factors (age, size, fur, breed, location etc) ('all else we can measure being equal'), it indeed grows a bit which tells us that black cats' longer wait isn't being caused by them being older or anything else, it really does seem tied to the colour itself.

## Verdict

**The penalty holds.** Black cats take ~15–17% longer to adopt, all else equal (median 23 vs 20 days,
n≈621k, p≈0). Robust to age/size/coat/breed/spay/state/year and even grows with controls. Age does not
confound; black cats pile up in the still-waiting pool (33% vs 28%).

## Findings summary

_Mirror of `../ANALYSIS_FINDINGS.md`. Analysis on `../data/adopted_and_adoptable_cats.csv` (841,776 adopted cats with valid `days_to_adopt`; Nevada via ZIP, 2020–2025 adoption flow)._

### Verdict: **YES — the penalty is real and robust to controls.**

Unlike The Pudding's black-**dog** finding (where the gap vanishes once you control for age/breed — the myth is composition), the black-**cat** penalty in this dataset **survives every control and even grows slightly**. This is the differentiated headline: *black-dog syndrome is a myth; black-cat syndrome, in this data, is not.*

### The four checks (in order of what could have invalidated it)

**1. The `days_to_adopt == 0` spike is an artifact — but a harmless one.**
13.5% of adopted cats sit at exactly 0 days. It is **concentrated in `Unknown`-color records** (43% of Unknown vs ~5% of Black and ~5% of Non-Black — equal). Since Unknown is excluded from the comparison and the 0-rate is identical for Black/Non-Black, the spike does **not** bias the finding. All models use `days_to_adopt >= 1`.

**2. Age does NOT explain the gap (the prime confounder fails to confound).**
Black cats are *not* older — their age mix is nearly identical to non-black (51.9% vs 49.7% Baby; 23.4% vs 25.7% Adult), if anything slightly *younger*. The median penalty **persists within every age band**: Adult +4d, Baby +3d, Senior +4d, Young +5d. Controlling for age *increases* the coefficient (younger cats should adopt faster, so adjusting for age reveals more color penalty).

**3. Black cats pile up in the still-waiting pool (model-independent evidence).**
Black share of known-color cats: **27.8% of adopted** vs **33.1% of still-waiting (adoptable)**. Overrepresented among cats still waiting by ~5 pts — direct corroboration independent of the days model, and it suggests the adopted-only estimate *understates* the true penalty.

**4. Regression — the coefficient does not shrink with controls.**
Outcome = `log1p(days_to_adopt)`, adopted cats, days≥1, Unknown excluded, n=621,218. Black = solid `Black` category.

| Model | Controls | Black effect | p |
|-------|----------|--------------|---|
| M0 | none (raw) | **+13.5%** longer | ~0 |
| M1 | + age | +15.1% | ~0 |
| M2 | + age, size, coat, breed-mixed, spay/neuter, state, list-year | +16.5% | ~0 |
| M3 | M2 + photo_count | +17.0% | ~0 |

Baseline medians: Black **23 days** vs Non-Black **20 days** (~+3–4 median days; larger in the mean/tail, ~+8–9 days on a ~50-day mean).

### Caveats to carry into the story (Pudding-grade honesty)
- **Effect size is modest but consistent.** ~+3 median days / +15–17% adjusted. Highly significant because n is huge — communicate the *real-days* magnitude, not just "significant."
- **`photo_count` is endogenous** — its coefficient is positive (more photos ↔ *longer* waits, i.e. shelters add photos to hard-to-place cats), so it's a collider/mediator, not a clean control. Black cats have essentially the same photo count (2.99 vs 3.06); the "black cats photograph poorly → fewer/worse photos" mechanism is **not** visible in photo *count* here (could still be photo *quality*, which we don't measure). Don't over-claim the photo angle.
- **Adopted-only survivorship**: `days_to_adopt` exists only for cats that got adopted. Check #3 shows the direction — the penalty is likely *understated*, not manufactured.
- **`adopted_year` is entirely null** for adopted rows (data-prep gap); used `published_year` as the time control instead. Worth fixing in prep.
- **Standard errors are optimistic** — cats are nested within shelters (`org_id`); clustered/robust SEs are the honest upgrade. The *coefficient* is trustworthy; the p-values are overstated.
- Cox proportional-hazards on adopted+adoptable (with censoring) is the rigorous upgrade for the methods section — but the snapshot adoptable pool is length-biased, so don't naively pool it.

### Reproduce
Run every cell above top-to-bottom with the anaconda `python3` kernel (`/opt/anaconda3/bin/python` — has `pandas` + `statsmodels`; base `python3` does not). Cells: load → Check 1 → Check 2 → Check 3 → Check 4 (regression).